# CBs and CB-Pairs Global Analyses <br> (Ground Duels + Aerial Duels + Ball Passing)

This notebook builds a **global cross-quality analysis layer** across the three CB quality families and the three existing assessment contexts:

1. **Single / individual CBs**
2. **CB pairs / duos**
3. **Anchor CB potential companion fits**

The workflow below keeps the original 9 source CSV files untouched and exports new global artifacts to:
`data/data-CB_pairing_AI_analyst/Qualities/0) Global Analysis`.

## Methodology and Locked Defaults

The global analysis follows these locked rules:

- **Cohort policy:** strict intersection across all 3 qualities.
- **Weighting policy:** equal weighting (`1/3 + 1/3 + 1/3`).
- **Radar basis:** quality z-score family (rank fields are preserved for interpretation and selection).
- **Source integrity:** original quality CSVs are never modified.

Expected intersection sizes with current data snapshot:

- Single CBs: **63**
- CB pair keys (unordered): **1953**
- Anchor-partner directional rows: **3906**

### Notes for Practical Usage

- Use the exported global CSVs as stable interfaces for downstream dashboards and model diagnostics.
- Use rank selectors (`CB_Rank`, `CB_Pair_Rank`, `Companion_Rank`) to produce repeatable benchmark plots.
- Use name/ID selectors for tactical analysis sessions and case-by-case scouting narratives.

### Library Imports

This notebook uses pandas/numpy for data assembly and the newly added global radar classes from `classes.visual_Sandbox` for visualization.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

import plotly.io as pio
pio.renderers.default = "notebook_connected"


import pathlib
import sys
sys.path.append(str(pathlib.Path().resolve().parents[0]))
from classes.visual_Sandbox import Single_CB_Global_Qualities_Radar_Plot, CB_Pair_Global_Qualities_Radar_Plot, Anchor_CB_Companion_Fit_Global_Qualities_Radar_Plot

## Paths and Source Loading

In [2]:
from pathlib import Path


def _resolve_repo_root(start_path: Path) -> Path:
    """
    Resolve the repository root even when the notebook is executed from a nested folder.

    Strategy:
    - Start from the current working directory used by the notebook kernel.
    - Walk upward through parent directories.
    - Pick the first directory containing the Qualities data folder.
    """
    required_csv_rel = (
        Path("data")
        / "data-CB_pairing_AI_analyst"
        / "Qualities"
        / "Ground Duels"
        / "CB_ground_duels.csv"
    )
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / required_csv_rel).exists():
            return candidate

    checked = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Could not locate the repository root from the current kernel working directory. "
        f"Searched these locations:\n{checked}"
    )


REPO_ROOT = _resolve_repo_root(Path.cwd())
QUALITIES_DIR = REPO_ROOT / "data" / "data-CB_pairing_AI_analyst" / "Qualities"
OUTPUT_DIR = QUALITIES_DIR / "0) Global Analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_paths = {
    "single_ground": QUALITIES_DIR / "Ground Duels" / "CB_ground_duels.csv",
    "single_aerial": QUALITIES_DIR / "Aerial Duels" / "CB_aerial_duels.csv",
    "single_ball": QUALITIES_DIR / "Ball Passing" / "CB_ball_passing.csv",
    "pair_ground": QUALITIES_DIR / "Ground Duels" / "CB_pairs_ground_duels.csv",
    "pair_aerial": QUALITIES_DIR / "Aerial Duels" / "CB_pairs_aerial_duels.csv",
    "pair_ball": QUALITIES_DIR / "Ball Passing" / "CB_pairs_ball_passing.csv",
    "comp_ground": QUALITIES_DIR / "Ground Duels" / "CB_companion_fits_ground_duels.csv",
    "comp_aerial": QUALITIES_DIR / "Aerial Duels" / "CB_companion_fits_aerial_duels.csv",
    "comp_ball": QUALITIES_DIR / "Ball Passing" / "CB_companion_fits_ball_passing.csv",
}

missing = {key: path for key, path in source_paths.items() if not path.exists()}
if missing:
    missing_lines = "\n".join(f"- {key}: {path}" for key, path in missing.items())
    raise FileNotFoundError(f"The following source CSV files were not found:\n{missing_lines}")

print(f"Resolved REPO_ROOT: {REPO_ROOT}")

data = {key: pd.read_csv(path) for key, path in source_paths.items()}

if "Unnamed: 0" in data["single_ground"].columns:
    data["single_ground"] = data["single_ground"].drop(columns=["Unnamed: 0"])

for key, df in data.items():
    print(f"{key:>13}: rows={len(df):>5}, cols={len(df.columns):>2}")

Resolved REPO_ROOT: c:\Users\Pedro\Desktop\Portfolio\12 Football CE - CB Pairing AI Analyst\Twelve-GPT-Educational-Giorgis
single_ground: rows=   74, cols=21
single_aerial: rows=   65, cols=21
  single_ball: rows=   88, cols=25
  pair_ground: rows= 2701, cols=36
  pair_aerial: rows= 2080, cols=36
    pair_ball: rows= 3828, cols=36
  comp_ground: rows= 5402, cols=10
  comp_aerial: rows= 4160, cols=10
    comp_ball: rows= 7656, cols=10


## Helper Functions

In [3]:
def rank_to_z_higher_is_better(rank_series: pd.Series) -> pd.Series:
    # Convert rank values (1=best) to a standardized scale where higher is better.
    ranks = pd.to_numeric(rank_series, errors="coerce").astype(float)
    std = ranks.std(ddof=0)
    if np.isnan(std) or std == 0:
        return pd.Series(np.zeros(len(ranks)), index=ranks.index, dtype=float)
    return (ranks.mean() - ranks) / std


def zscore_within_group(series: pd.Series) -> pd.Series:
    # Standardize a metric within one group; use zeros for constant groups.
    s = pd.to_numeric(series, errors="coerce").astype(float)
    std = s.std(ddof=0)
    if np.isnan(std) or std == 0:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    return (s - s.mean()) / std

## Build Global Table 1: Single CBs

In [4]:
single_ground = data["single_ground"][[
    "player.id",
    "player.name",
    "CB_ground_duels_quality_z_score",
    "CB_rank_for_ground_duels_quality",
    "CB_rank_for_ground_duels_quality_z_score",
]].copy()

single_aerial = data["single_aerial"][[
    "player.id",
    "player.name",
    "CB_aerial_duels_quality_z_score",
    "CB_rank_for_aerial_duels_quality",
    "CB_rank_for_aerial_duels_quality_z_score",
]].copy()

single_ball = data["single_ball"][[
    "player.id",
    "player.name",
    "CB_ball_passing_quality_z_score",
    "CB_rank_for_ball_passing_quality",
    "CB_rank_for_ball_passing_quality_z_score",
]].copy()

single_ids_intersection = (
    set(single_ground["player.id"]) 
    & set(single_aerial["player.id"]) 
    & set(single_ball["player.id"])
)

single_ground = single_ground[single_ground["player.id"].isin(single_ids_intersection)].copy()
single_aerial = single_aerial[single_aerial["player.id"].isin(single_ids_intersection)].copy()
single_ball = single_ball[single_ball["player.id"].isin(single_ids_intersection)].copy()

single_ground = single_ground.rename(columns={"player.name": "player.name_ground"})
single_aerial = single_aerial.rename(columns={"player.name": "player.name_aerial"})
single_ball = single_ball.rename(columns={"player.name": "player.name_ball"})

df_global_single = (
    single_ground
    .merge(single_aerial, on="player.id", how="inner", validate="one_to_one")
    .merge(single_ball, on="player.id", how="inner", validate="one_to_one")
)

df_global_single["player.name"] = (
    df_global_single["player.name_ball"]
    .fillna(df_global_single["player.name_ground"])
    .fillna(df_global_single["player.name_aerial"])
)

df_global_single = df_global_single[[
    "player.id",
    "player.name",
    "CB_ground_duels_quality_z_score",
    "CB_rank_for_ground_duels_quality",
    "CB_rank_for_ground_duels_quality_z_score",
    "CB_aerial_duels_quality_z_score",
    "CB_rank_for_aerial_duels_quality",
    "CB_rank_for_aerial_duels_quality_z_score",
    "CB_ball_passing_quality_z_score",
    "CB_rank_for_ball_passing_quality",
    "CB_rank_for_ball_passing_quality_z_score",
]].copy()

df_global_single["global_quality_z_score"] = df_global_single[[
    "CB_ground_duels_quality_z_score",
    "CB_aerial_duels_quality_z_score",
    "CB_ball_passing_quality_z_score",
]].mean(axis=1)

df_global_single["global_quality_rank"] = (
    df_global_single["global_quality_z_score"].rank(ascending=False, method="min").astype(int)
)

df_global_single["global_quality_rank_z_score"] = rank_to_z_higher_is_better(
    df_global_single["global_quality_rank"]
)

df_global_single = df_global_single.sort_values(
    ["global_quality_rank", "player.name"],
    ascending=[True, True],
).reset_index(drop=True)

df_global_single.head(8)

,player.id,player.name,CB_ground_duels_quality_z_score,CB_rank_for_ground_duels_quality,CB_rank_for_ground_duels_quality_z_score,CB_aerial_duels_quality_z_score,CB_rank_for_aerial_duels_quality,CB_rank_for_aerial_duels_quality_z_score,CB_ball_passing_quality_z_score,CB_rank_for_ball_passing_quality,CB_rank_for_ball_passing_quality_z_score,global_quality_z_score,global_quality_rank,global_quality_rank_z_score
0,568412,J. Gvardiol,2.244289,1,1.697216,0.559930,23,0.528886,2.873593,1,1.712479,1.892604,1,1.704773
1,370,V. van Dijk,1.369088,4,1.557718,2.370236,1,1.692435,1.699381,4,1.594377,1.812902,2,1.649780
2,578325,W. Saliba,0.575618,24,0.627737,1.172248,7,1.375103,1.197278,12,1.279438,0.981715,3,1.594787
3,590781,J. van Hecke,-0.069308,42,-0.209246,1.233674,5,1.480880,1.651687,5,1.555010,0.938684,4,1.539795
4,623641,E. Agbadou,1.990972,3,1.604217,1.082164,8,1.322215,-0.259026,55,-0.413357,0.938037,5,1.484802
5,393247,I. Konaté,0.268108,32,0.255745,1.632285,3,1.586658,0.670761,23,0.846398,0.857051,6,1.429809
6,349142,N. Mazraoui,2.080873,2,1.650716,-0.593292,44,-0.581774,0.794437,19,1.003867,0.760673,7,1.374817
7,62126,M. Doherty,0.030402,39,-0.069749,1.734107,2,1.639546,0.195949,36,0.334622,0.653486,8,1.319824


## Build Global Table 2: CB Pairs

In [5]:
pair_metadata_cols = [
    "pair_key",
    "pair_name",
    "player.id_CB1",
    "player.name_CB1",
    "player.id_CB2",
    "player.name_CB2",
    "team_CB1_id",
    "team_CB2_id",
    "same_dominant_team",
    "CB_dominant_playing_side_CB1",
    "CB_dominant_playing_side_CB2",
    "opposite_side_pair",
    "shared_matches",
    "shared_minutes_overlap",
    "evidence_band",
]

pair_score_cols = [
    "quality_raw",
    "complement_raw",
    "floor_raw",
    "quality_z",
    "complement_z",
    "floor_z",
    "CB_pair_fit_z_score",
    "CB_pair_fit_rank",
    "CB_pair_fit_rank_z_score",
]

pair_ball = data["pair_ball"][pair_metadata_cols + pair_score_cols].copy()
pair_ground = data["pair_ground"][["pair_key"] + pair_score_cols].copy()
pair_aerial = data["pair_aerial"][["pair_key"] + pair_score_cols].copy()

for frame_name, frame in [("pair_ball", pair_ball), ("pair_ground", pair_ground), ("pair_aerial", pair_aerial)]:
    if not frame["pair_key"].is_unique:
        raise ValueError(f"{frame_name} contains duplicate pair_key values.")

pair_keys_intersection = (
    set(pair_ball["pair_key"]) 
    & set(pair_ground["pair_key"]) 
    & set(pair_aerial["pair_key"])
)

pair_ball = pair_ball[pair_ball["pair_key"].isin(pair_keys_intersection)].copy()
pair_ground = pair_ground[pair_ground["pair_key"].isin(pair_keys_intersection)].copy()
pair_aerial = pair_aerial[pair_aerial["pair_key"].isin(pair_keys_intersection)].copy()

pair_ball = pair_ball.rename(columns={col: f"ball_{col}" for col in pair_score_cols})
pair_ground = pair_ground.rename(columns={col: f"ground_{col}" for col in pair_score_cols})
pair_aerial = pair_aerial.rename(columns={col: f"aerial_{col}" for col in pair_score_cols})

df_global_pairs = (
    pair_ball
    .merge(pair_ground, on="pair_key", how="inner", validate="one_to_one")
    .merge(pair_aerial, on="pair_key", how="inner", validate="one_to_one")
)

df_global_pairs["global_CB_pair_fit_z_score"] = df_global_pairs[[
    "ground_CB_pair_fit_z_score",
    "aerial_CB_pair_fit_z_score",
    "ball_CB_pair_fit_z_score",
]].mean(axis=1)

df_global_pairs["global_CB_pair_fit_rank"] = (
    df_global_pairs["global_CB_pair_fit_z_score"].rank(ascending=False, method="min").astype(int)
)

df_global_pairs["global_CB_pair_fit_rank_z_score"] = rank_to_z_higher_is_better(
    df_global_pairs["global_CB_pair_fit_rank"]
)

df_global_pairs = df_global_pairs.sort_values(
    ["global_CB_pair_fit_rank", "pair_name"],
    ascending=[True, True],
).reset_index(drop=True)

df_global_pairs.head(8)

,pair_key,pair_name,player.id_CB1,player.name_CB1,player.id_CB2,player.name_CB2,team_CB1_id,team_CB2_id,same_dominant_team,CB_dominant_playing_side_CB1,...,aerial_floor_raw,aerial_quality_z,aerial_complement_z,aerial_floor_z,aerial_CB_pair_fit_z_score,aerial_CB_pair_fit_rank,aerial_CB_pair_fit_rank_z_score,global_CB_pair_fit_z_score,global_CB_pair_fit_rank,global_CB_pair_fit_rank_z_score
0,370__568412,V. van Dijk + J. Gvardiol,370,V. van Dijk,568412,J. Gvardiol,1612,1625,False,left,...,0.359735,2.104066,0.273679,1.647283,1.463593,24,1.692506,1.681199,1,1.731164
1,370__349142,V. van Dijk + N. Mazraoui,370,V. van Dijk,349142,N. Mazraoui,1612,1611,False,left,...,-0.381169,1.275971,1.866234,0.256586,1.249173,56,1.639225,1.432237,2,1.729390
2,370__412968,V. van Dijk + M. Senesi,370,V. van Dijk,412968,M. Senesi,1612,1659,False,left,...,-0.102268,1.598779,1.288062,0.780091,1.341826,42,1.662536,1.426359,3,1.727617
3,568412__578325,J. Gvardiol + W. Saliba,568412,J. Gvardiol,578325,W. Saliba,1625,1609,False,left,...,0.208177,1.243825,-0.729155,1.362804,0.675727,292,1.246277,1.368930,4,1.725843
4,349142__568412,N. Mazraoui + J. Gvardiol,349142,N. Mazraoui,568412,J. Gvardiol,1611,1625,False,right,...,-0.397463,-0.023956,-0.563682,0.226002,-0.135882,1306,-0.442066,1.361008,5,1.724069
5,412968__568412,M. Senesi + J. Gvardiol,412968,M. Senesi,568412,J. Gvardiol,1659,1625,False,left,...,-0.361357,0.298852,-0.098087,0.293773,0.178754,852,0.313859,1.360437,6,1.722295
6,568412__623641,J. Gvardiol + E. Agbadou,568412,J. Gvardiol,623641,E. Agbadou,1625,1629,False,left,...,0.331328,1.179139,-1.382980,1.593963,0.493468,469,0.951567,1.357596,7,1.720522
7,393247__568412,I. Konaté + J. Gvardiol,393247,I. Konaté,568412,J. Gvardiol,1612,1625,False,right,...,0.359735,1.574164,-0.745403,1.647283,0.892918,155,1.474387,1.330483,8,1.718748


## Build Global Table 3: Anchor Companion Fits (Directional)

In [6]:
comp_id_cols = [
    "anchor_player_id",
    "anchor_player_name",
    "partner_player_id",
    "partner_player_name",
    "pair_key",
]
comp_score_cols = [
    "companion_fit_score",
    "companion_rank_for_anchor",
    "CB_pair_fit_z_score",
    "coverage_gain_raw",
    "coverage_gain_z_within_anchor",
]

comp_ball = data["comp_ball"][comp_id_cols + comp_score_cols].copy()
comp_ground = data["comp_ground"][comp_id_cols + comp_score_cols].copy()
comp_aerial = data["comp_aerial"][comp_id_cols + comp_score_cols].copy()

for frame_name, frame in [("comp_ball", comp_ball), ("comp_ground", comp_ground), ("comp_aerial", comp_aerial)]:
    key_dupes = frame.duplicated(["anchor_player_id", "partner_player_id"]).sum()
    if key_dupes:
        raise ValueError(f"{frame_name} has {key_dupes} duplicated directional keys.")

directional_key_ball = set(zip(comp_ball["anchor_player_id"], comp_ball["partner_player_id"]))
directional_key_ground = set(zip(comp_ground["anchor_player_id"], comp_ground["partner_player_id"]))
directional_key_aerial = set(zip(comp_aerial["anchor_player_id"], comp_aerial["partner_player_id"]))
directional_intersection = directional_key_ball & directional_key_ground & directional_key_aerial

def filter_directional(df: pd.DataFrame) -> pd.DataFrame:
    keys = list(zip(df["anchor_player_id"], df["partner_player_id"]))
    mask = [k in directional_intersection for k in keys]
    return df.loc[mask].copy()

comp_ball = filter_directional(comp_ball)
comp_ground = filter_directional(comp_ground)
comp_aerial = filter_directional(comp_aerial)

comp_ball = comp_ball.rename(columns={
    "companion_fit_score": "ball_companion_fit_score",
    "companion_rank_for_anchor": "ball_companion_rank_for_anchor",
    "CB_pair_fit_z_score": "ball_CB_pair_fit_z_score",
    "coverage_gain_raw": "ball_coverage_gain_raw",
    "coverage_gain_z_within_anchor": "ball_coverage_gain_z_within_anchor",
})

comp_ground = comp_ground.rename(columns={
    "anchor_player_name": "anchor_player_name_ground",
    "partner_player_name": "partner_player_name_ground",
    "pair_key": "pair_key_ground",
    "companion_fit_score": "ground_companion_fit_score",
    "companion_rank_for_anchor": "ground_companion_rank_for_anchor",
    "CB_pair_fit_z_score": "ground_CB_pair_fit_z_score",
    "coverage_gain_raw": "ground_coverage_gain_raw",
    "coverage_gain_z_within_anchor": "ground_coverage_gain_z_within_anchor",
})

comp_aerial = comp_aerial.rename(columns={
    "anchor_player_name": "anchor_player_name_aerial",
    "partner_player_name": "partner_player_name_aerial",
    "pair_key": "pair_key_aerial",
    "companion_fit_score": "aerial_companion_fit_score",
    "companion_rank_for_anchor": "aerial_companion_rank_for_anchor",
    "CB_pair_fit_z_score": "aerial_CB_pair_fit_z_score",
    "coverage_gain_raw": "aerial_coverage_gain_raw",
    "coverage_gain_z_within_anchor": "aerial_coverage_gain_z_within_anchor",
})

merge_keys = ["anchor_player_id", "partner_player_id"]
df_global_companion = (
    comp_ball
    .merge(comp_ground.drop(columns=["anchor_player_name_ground", "partner_player_name_ground", "pair_key_ground"]), on=merge_keys, how="inner", validate="one_to_one")
    .merge(comp_aerial.drop(columns=["anchor_player_name_aerial", "partner_player_name_aerial", "pair_key_aerial"]), on=merge_keys, how="inner", validate="one_to_one")
)

df_global_companion["ground_companion_fit_score_z_within_anchor"] = (
    df_global_companion.groupby("anchor_player_id")["ground_companion_fit_score"].transform(zscore_within_group)
)
df_global_companion["aerial_companion_fit_score_z_within_anchor"] = (
    df_global_companion.groupby("anchor_player_id")["aerial_companion_fit_score"].transform(zscore_within_group)
)
df_global_companion["ball_companion_fit_score_z_within_anchor"] = (
    df_global_companion.groupby("anchor_player_id")["ball_companion_fit_score"].transform(zscore_within_group)
)

df_global_companion["global_companion_fit_score_z_within_anchor"] = df_global_companion[[
    "ground_companion_fit_score_z_within_anchor",
    "aerial_companion_fit_score_z_within_anchor",
    "ball_companion_fit_score_z_within_anchor",
]].mean(axis=1)

df_global_companion["global_companion_rank_for_anchor"] = (
    df_global_companion
    .groupby("anchor_player_id")["global_companion_fit_score_z_within_anchor"]
    .rank(ascending=False, method="min")
    .astype(int)
)

df_global_companion["global_companion_rank_for_anchor_z_score"] = (
    df_global_companion
    .groupby("anchor_player_id")["global_companion_rank_for_anchor"]
    .transform(rank_to_z_higher_is_better)
)

output_order = [
    "anchor_player_id",
    "anchor_player_name",
    "partner_player_id",
    "partner_player_name",
    "pair_key",
    "ball_companion_fit_score",
    "ball_companion_rank_for_anchor",
    "ball_CB_pair_fit_z_score",
    "ball_coverage_gain_raw",
    "ball_coverage_gain_z_within_anchor",
    "ground_companion_fit_score",
    "ground_companion_rank_for_anchor",
    "ground_CB_pair_fit_z_score",
    "ground_coverage_gain_raw",
    "ground_coverage_gain_z_within_anchor",
    "aerial_companion_fit_score",
    "aerial_companion_rank_for_anchor",
    "aerial_CB_pair_fit_z_score",
    "aerial_coverage_gain_raw",
    "aerial_coverage_gain_z_within_anchor",
    "ground_companion_fit_score_z_within_anchor",
    "aerial_companion_fit_score_z_within_anchor",
    "ball_companion_fit_score_z_within_anchor",
    "global_companion_fit_score_z_within_anchor",
    "global_companion_rank_for_anchor",
    "global_companion_rank_for_anchor_z_score",
]

df_global_companion = df_global_companion[output_order].sort_values(
    ["anchor_player_id", "global_companion_rank_for_anchor", "partner_player_name"],
    ascending=[True, True, True],
).reset_index(drop=True)

df_global_companion.head(8)

,anchor_player_id,anchor_player_name,partner_player_id,partner_player_name,pair_key,ball_companion_fit_score,ball_companion_rank_for_anchor,ball_CB_pair_fit_z_score,ball_coverage_gain_raw,ball_coverage_gain_z_within_anchor,...,aerial_companion_rank_for_anchor,aerial_CB_pair_fit_z_score,aerial_coverage_gain_raw,aerial_coverage_gain_z_within_anchor,ground_companion_fit_score_z_within_anchor,aerial_companion_fit_score_z_within_anchor,ball_companion_fit_score_z_within_anchor,global_companion_fit_score_z_within_anchor,global_companion_rank_for_anchor,global_companion_rank_for_anchor_z_score
0,370,V. van Dijk,568412,J. Gvardiol,370__568412,2.246647,1,2.088478,0.163913,2.540390,...,23,1.463593,0.0,0.0,1.280161,0.528047,3.134353,1.647520,1,1.704336
1,370,V. van Dijk,412968,M. Senesi,370__412968,1.613292,5,1.377667,0.139052,2.050882,...,36,1.341826,0.0,0.0,2.259039,-0.110150,1.954422,1.367770,2,1.648456
2,370,V. van Dijk,349142,N. Mazraoui,370__349142,1.498927,8,1.201720,0.139052,2.050882,...,43,1.249173,0.0,0.0,2.622774,-0.595762,1.741361,1.256124,3,1.592576
3,370,V. van Dijk,578325,W. Saliba,370__578325,1.164185,14,1.290673,0.082090,0.929279,...,2,1.686624,0.0,0.0,0.542542,1.696985,1.117743,1.119090,4,1.536697
4,370,V. van Dijk,777938,Dean Huijsen,370__777938,1.476358,9,1.112702,0.144173,2.151718,...,22,1.465696,0.0,0.0,0.958309,0.539068,1.699315,1.065564,5,1.480817
5,370,V. van Dijk,240032,J. Bednarek,370__240032,0.914932,20,0.907207,0.082090,0.929279,...,16,1.515445,0.0,0.0,1.636563,0.799811,0.653388,1.029921,6,1.424937
6,370,V. van Dijk,590781,J. van Hecke,370__590781,1.910869,2,1.427014,0.177577,2.809457,...,9,1.588863,0.0,0.0,-0.643581,1.184607,2.508804,1.016610,7,1.369057
7,370,V. van Dijk,590703,I. Zabarnyi,370__590703,0.975552,19,0.960260,0.085882,1.003950,...,29,1.423677,0.0,0.0,1.660156,0.318841,0.766321,0.915106,8,1.313177


## Export Global CSV Artifacts

In [7]:
output_single_path = OUTPUT_DIR / "CBs_global_qualities.csv"
output_pair_path = OUTPUT_DIR / "CB_pairs_global_qualities.csv"
output_companion_path = OUTPUT_DIR / "CB_companion_fits_global_qualities.csv"

df_global_single.to_csv(output_single_path, index=False)
df_global_pairs.to_csv(output_pair_path, index=False)
df_global_companion.to_csv(output_companion_path, index=False)

print("Exported:")
print(f"- {output_single_path}")
print(f"- {output_pair_path}")
print(f"- {output_companion_path}")

Exported:
- c:\Users\Pedro\Desktop\Portfolio\12 Football CE - CB Pairing AI Analyst\Twelve-GPT-Educational-Giorgis\data\data-CB_pairing_AI_analyst\Qualities\0) Global Analysis\CBs_global_qualities.csv
- c:\Users\Pedro\Desktop\Portfolio\12 Football CE - CB Pairing AI Analyst\Twelve-GPT-Educational-Giorgis\data\data-CB_pairing_AI_analyst\Qualities\0) Global Analysis\CB_pairs_global_qualities.csv
- c:\Users\Pedro\Desktop\Portfolio\12 Football CE - CB Pairing AI Analyst\Twelve-GPT-Educational-Giorgis\data\data-CB_pairing_AI_analyst\Qualities\0) Global Analysis\CB_companion_fits_global_qualities.csv


## Sanity Checks and Data Validation

In [8]:
expected_counts = {
    "single": 63,
    "pairs": 1953,
    "companion": 3906,
}

actual_counts = {
    "single": len(df_global_single),
    "pairs": len(df_global_pairs),
    "companion": len(df_global_companion),
}

print("Row counts:", actual_counts)
for key in expected_counts:
    assert actual_counts[key] == expected_counts[key], (
        f"Unexpected count for {key}: expected {expected_counts[key]}, got {actual_counts[key]}"
    )

required_single = ["player.id", "player.name", "global_quality_z_score", "global_quality_rank"]
required_pairs = ["pair_key", "pair_name", "global_CB_pair_fit_z_score", "global_CB_pair_fit_rank"]
required_comp = [
    "anchor_player_id",
    "partner_player_id",
    "global_companion_fit_score_z_within_anchor",
    "global_companion_rank_for_anchor",
]

assert not df_global_single[required_single].isna().any().any(), "Nulls detected in single global required columns."
assert not df_global_pairs[required_pairs].isna().any().any(), "Nulls detected in pair global required columns."
assert not df_global_companion[required_comp].isna().any().any(), "Nulls detected in companion global required columns."

single_rank_ties = int(df_global_single.duplicated(["global_quality_rank"]).sum())
pair_rank_ties = int(df_global_pairs.duplicated(["global_CB_pair_fit_rank"]).sum())
comp_rank_ties_total = int(
    df_global_companion.groupby("anchor_player_id").apply(
        lambda x: x.duplicated(["global_companion_rank_for_anchor"]).sum()
    ).sum()
)

print("Rank tie diagnostics:")
print(f"- single global rank ties: {single_rank_ties}")
print(f"- pair global rank ties: {pair_rank_ties}")
print(f"- companion global rank ties across anchors: {comp_rank_ties_total}")

print("All validation checks passed.")

Row counts: {'single': 63, 'pairs': 1953, 'companion': 3906}
Rank tie diagnostics:
- single global rank ties: 0
- pair global rank ties: 0
- companion global rank ties across anchors: 0
All validation checks passed.


C:\Users\Pedro\AppData\Local\Temp\ipykernel_36468\431868806.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_global_companion.groupby("anchor_player_id").apply(


## Global Radar Analysis: Single CBs

Radar axis categories are fixed to:

- Ground Duels
- Aerial Duels
- Ball Passing

For this context, the radar radius uses each quality's `CB_*_quality_z_score`.

In [9]:
single_radar = Single_CB_Global_Qualities_Radar_Plot(df_global_single_cb=df_global_single)

top_single_names = (
    df_global_single.sort_values("global_quality_rank")["player.name"].head(10).tolist()
)
print("Top individual CBs by Global Quality ranking:")
print([f"{i+1}) {CB_Name}" for i, CB_Name in enumerate(top_single_names)])

single_radar.Plot_Single_CB(
    CB=None,
    CB_Rank = None,
    include_best=True,
    include_worst=True,
    include_league_average=True,
    radial_range = None,
    show=True,
)

Top individual CBs by Global Quality ranking:
['1) J. Gvardiol', '2) V. van Dijk', '3) W. Saliba', '4) J. van Hecke', '5) E. Agbadou', '6) I. Konaté', '7) N. Mazraoui', '8) M. Doherty', '9) Dean Huijsen', '10) I. Zabarnyi']


In [16]:
single_radar.Plot_CBs_Comparison(
    CBs=top_single_names[:5],
    CB_Ranks = None,
    include_league_average=True,
    include_best=False,
    include_worst=False,
    radial_range = None,
    show=True,
)

## Global Radar Analysis: CB Pairs

For pair analysis, radar radii are the three per-quality `*_CB_pair_fit_z_score` values.

In [11]:
pair_radar = CB_Pair_Global_Qualities_Radar_Plot(df_global_cb_pairs=df_global_pairs)

top_pair_names = (
    df_global_pairs.sort_values("global_CB_pair_fit_rank")["pair_name"].head(3).tolist()
)
print("Top CB pairs by global rank:", top_pair_names)

pair_radar.Plot_CB_Pair(
    CB_Pair=top_pair_names[0],
    CB_1 = None,
    CB_2 = None,
    CB_Pair_Rank = None,
    include_best=True,
    include_worst=True,
    include_average=True,
    radial_range = None,
    show=True,
)

Top CB pairs by global rank: ['V. van Dijk + J. Gvardiol', 'V. van Dijk + N. Mazraoui', 'V. van Dijk + M. Senesi']


In [12]:
pair_radar.Plot_CB_Pairs_Comparison(
    CB_Pairs=top_pair_names,
    CB_Pair_Ranks = None,
    include_best=False,
    include_worst=False,
    include_average=True,
    radial_range = None,
    show=True,
)

## Global Radar Analysis: Anchor CB Companion Fits

For companion analysis, each axis uses the per-quality within-anchor z-score columns:

- `ground_companion_fit_score_z_within_anchor`
- `aerial_companion_fit_score_z_within_anchor`
- `ball_companion_fit_score_z_within_anchor`

In [13]:
companion_radar = Anchor_CB_Companion_Fit_Global_Qualities_Radar_Plot(
    df_global_companion_fits=df_global_companion
)

demo_anchor_id = (
    df_global_companion["anchor_player_id"].value_counts().sort_values(ascending=False).index[0]
)
demo_anchor_name = (
    df_global_companion.loc[
        df_global_companion["anchor_player_id"] == demo_anchor_id,
        "anchor_player_name",
    ].iloc[0]
)
print(f"Demo Anchor CB  →  {demo_anchor_name} (ID = {demo_anchor_id})")

companion_radar.Initialize_Desired_Anchor_CB(Anchor_CB=demo_anchor_name)

companion_radar.Plot_Companion_of_Anchor_CB(
    Anchor_CB=demo_anchor_name,
    Companion_CB = None,
    Companion_Rank = None,
    Companion_Ranks=[1, 2, 3],
    include_best=False,
    include_worst=False,
    include_anchor_CB_pool_average=True,
    top_n = None,
    radial_range = None,
    show=True,
)

Demo Anchor CB  →  V. van Dijk (ID = 370)


In [14]:
companion_radar.Plot_Companion_of_Anchor_CB(
    Anchor_CB=demo_anchor_name,
    Companion_CB = None,
    Companion_Rank = None,
    Companion_Ranks=None,
    include_best=False,
    include_worst=False,
    include_anchor_CB_pool_average=False,
    top_n = 5,
    radial_range = None,
    show=True,
)

## Interpretation Tables and Key Takeaways

In [15]:
top_single_table = df_global_single[[
    "global_quality_rank",
    "player.name",
    "CB_ground_duels_quality_z_score",
    "CB_aerial_duels_quality_z_score",
    "CB_ball_passing_quality_z_score",
    "global_quality_z_score",
]].sort_values("global_quality_rank").head(10)

top_pair_table = df_global_pairs[[
    "global_CB_pair_fit_rank",
    "pair_name",
    "ground_CB_pair_fit_z_score",
    "aerial_CB_pair_fit_z_score",
    "ball_CB_pair_fit_z_score",
    "global_CB_pair_fit_z_score",
    "evidence_band",
]].sort_values("global_CB_pair_fit_rank").head(10)

top_companion_table = df_global_companion.loc[
    df_global_companion["anchor_player_id"] == demo_anchor_id,
    [
        "global_companion_rank_for_anchor",
        "partner_player_name",
        "ground_companion_fit_score_z_within_anchor",
        "aerial_companion_fit_score_z_within_anchor",
        "ball_companion_fit_score_z_within_anchor",
        "global_companion_fit_score_z_within_anchor",
    ],
].sort_values("global_companion_rank_for_anchor").head(10)

display(top_single_table)
display(top_pair_table)
display(top_companion_table)

,global_quality_rank,player.name,CB_ground_duels_quality_z_score,CB_aerial_duels_quality_z_score,CB_ball_passing_quality_z_score,global_quality_z_score
0,1,J. Gvardiol,2.244289,0.559930,2.873593,1.892604
1,2,V. van Dijk,1.369088,2.370236,1.699381,1.812902
2,3,W. Saliba,0.575618,1.172248,1.197278,0.981715
3,4,J. van Hecke,-0.069308,1.233674,1.651687,0.938684
4,5,E. Agbadou,1.990972,1.082164,-0.259026,0.938037
5,6,I. Konaté,0.268108,1.632285,0.670761,0.857051
6,7,N. Mazraoui,2.080873,-0.593292,0.794437,0.760673
7,8,M. Doherty,0.030402,1.734107,0.195949,0.653486
8,9,Dean Huijsen,0.584089,0.571240,0.632799,0.596043
9,10,I. Zabarnyi,0.988602,0.345249,0.289110,0.540987


,global_CB_pair_fit_rank,pair_name,ground_CB_pair_fit_z_score,aerial_CB_pair_fit_z_score,ball_CB_pair_fit_z_score,global_CB_pair_fit_z_score,evidence_band
0,1,V. van Dijk + J. Gvardiol,1.491526,1.463593,2.088478,1.681199,none
1,2,V. van Dijk + N. Mazraoui,1.845817,1.249173,1.201720,1.432237,none
2,3,V. van Dijk + M. Senesi,1.559583,1.341826,1.377667,1.426359,none
3,4,J. Gvardiol + W. Saliba,1.327121,0.675727,2.103943,1.368930,none
4,5,N. Mazraoui + J. Gvardiol,2.338232,-0.135882,1.880675,1.361008,none
5,6,M. Senesi + J. Gvardiol,2.051998,0.178754,1.850559,1.360437,none
6,7,J. Gvardiol + E. Agbadou,1.930752,0.493468,1.648569,1.357596,none
7,8,I. Konaté + J. Gvardiol,1.245107,0.892918,1.853426,1.330483,none
8,9,M. Doherty + J. Gvardiol,1.174990,0.971660,1.748812,1.298487,none
9,10,J. Gvardiol + J. van Hecke,1.151413,0.619899,2.107937,1.293083,none


,global_companion_rank_for_anchor,partner_player_name,ground_companion_fit_score_z_within_anchor,aerial_companion_fit_score_z_within_anchor,ball_companion_fit_score_z_within_anchor,global_companion_fit_score_z_within_anchor
0,1,J. Gvardiol,1.280161,0.528047,3.134353,1.647520
1,2,M. Senesi,2.259039,-0.110150,1.954422,1.367770
2,3,N. Mazraoui,2.622774,-0.595762,1.741361,1.256124
3,4,W. Saliba,0.542542,1.696985,1.117743,1.119090
4,5,Dean Huijsen,0.958309,0.539068,1.699315,1.065564
5,6,J. Bednarek,1.636563,0.799811,0.653388,1.029921
6,7,J. van Hecke,-0.643581,1.184607,2.508804,1.016610
7,8,I. Zabarnyi,1.660156,0.318841,0.766321,0.915106
8,9,J. Andersen,-0.837459,0.846516,2.426440,0.811832
9,10,C. Romero,1.054367,-1.156727,1.923565,0.607068
